In [1]:
print("hello world")

hello world


In [23]:
# import libraries
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
import numpy as np

In [7]:
titanic = fetch_openml(name="titanic", as_frame=True, version=1)
data =pd.DataFrame(titanic.data)

data.head()

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\datasets\_openml.py:968: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\datasets\_arff_parser.py:200: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  frame = pd.concat(dfs, ignore_index=True)


,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1.0,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1.0,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1.0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,S,None,NaN,"Montreal, PQ / Chesterville, ON"
3,1.0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,S,None,135.0,"Montreal, PQ / Chesterville, ON"
4,1.0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,S,None,NaN,"Montreal, PQ / Chesterville, ON"


In [ ]:
# 1. Check for missing values
print("Missing values per column:")
print(data.isnull().sum())

# Display rows containing missing values
print("\nRows with missing values:")
print(data[data.isnull().any(axis=1)])

Missing values per column:
pclass          0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

Rows with missing values:
      pclass                                             name     sex  \
0        1.0                    Allen, Miss. Elisabeth Walton  female   
1        1.0                   Allison, Master. Hudson Trevor    male   
2        1.0                     Allison, Miss. Helen Loraine  female   
3        1.0             Allison, Mr. Hudson Joshua Creighton    male   
4        1.0  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female   
...      ...                                              ...     ...   
1304     3.0                             Zabour, Miss. Hileni  female   
1305     3.0                            Zabour, Miss. Thamine  female   
1306     3.0                        Zakari

In [9]:
# Fill numeric columns with the median
numeric_cols = data.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    data[col].fillna(data[col].median(), inplace=True)

# Fill categorical columns with the mode
categorical_cols = data.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    data[col].fillna(data[col].mode()[0], inplace=True)

# Verify there are no missing values left
print("\nMissing values after cleaning:")
print(data.isnull().sum())


Missing values after cleaning:
pclass       0
name         0
sex          0
age          0
sibsp        0
parch        0
ticket       0
fare         0
cabin        0
embarked     2
boat         0
body         0
home.dest    0
dtype: int64


In [ ]:
# 2. Check for duplicate rows
duplicate_count = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

# Display duplicate rows (optional)
print("\nDuplicate rows:")
print(data[data.duplicated()])

# Remove duplicate rows
data = data.drop_duplicates()

# Verify duplicates have been removed
print(f"\nRows after removing duplicates: {len(data)}")
print(f"Remaining duplicate rows: {data.duplicated().sum()}")

Number of duplicate rows: 0

Duplicate rows:
Empty DataFrame
Columns: [pclass, name, sex, age, sibsp, parch, ticket, fare, cabin, embarked, boat, body, home.dest]
Index: []

Rows after removing duplicates: 1309
Remaining duplicate rows: 0


In [ ]:
# Inspect the sex column for inconsistencies and standardize values

# View unique values in the Sex column
print(data["sex"].unique())

# Count occurrences of each value
print(data["sex"].value_counts(dropna=False))

# Standardize values in the Sex column
data["sex"] = (
    data["sex"]
    .str.strip()          # Remove leading/trailing spaces
    .str.lower()          # Convert to lowercase
)

# Replace inconsistent values
data["sex"] = data["sex"].replace({
    "m": "Male",
    "male": "Male",
    "f": "Female",
    "female": "Female"
})

# Check the results
print(data["sex"].value_counts())


['female' 'male']
sex
male      843
female    466
Name: count, dtype: int64
sex
Male      843
Female    466
Name: count, dtype: int64


In [19]:
# Identify outliers without removing them

for col in ["age", "fare"]:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    print(f"{col} outliers:")
    print(data[(data[col] < lower) | (data[col] > upper)])

age outliers:
      pclass                                  name     sex      age  sibsp  \
1        1.0        Allison, Master. Hudson Trevor    Male   0.9167    1.0   
2        1.0          Allison, Miss. Helen Loraine  Female   2.0000    1.0   
6        1.0     Andrews, Miss. Kornelia Theodosia  Female  63.0000    1.0   
9        1.0               Artagaveytia, Mr. Ramon    Male  71.0000    0.0   
14       1.0  Barkworth, Mr. Algernon Henry Wilson    Male  80.0000    0.0   
...      ...                                   ...     ...      ...    ...   
1225     3.0                    Storey, Mr. Thomas    Male  60.5000    0.0   
1230     3.0            Strom, Miss. Telma Matilda  Female   2.0000    0.0   
1235     3.0                   Svensson, Mr. Johan    Male  74.0000    0.0   
1240     3.0       Thomas, Master. Assad Alexander    Male   0.4167    0.0   
1261     3.0                Turkula, Mrs. (Hedwig)  Female  63.0000    0.0   

      parch    ticket      fare        cabin emba

In [21]:
# Create a new Family_Size feature
data["family_size"] = data["sibsp"] + data["parch"] + 1

# Display the first few rows
print(data[["sibsp", "parch", "family_size"]].head())

print(data["family_size"].describe())


   sibsp  parch  family_size
0    0.0    0.0          1.0
1    1.0    2.0          4.0
2    1.0    2.0          4.0
3    1.0    2.0          4.0
4    1.0    2.0          4.0
count    1309.000000
mean        1.883881
std         1.583639
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        11.000000
Name: family_size, dtype: float64


In [25]:
# Create a LabelEncoder object
le = LabelEncoder()

# Convert the Embarked column to numerical values
data["embarked"] = le.fit_transform(data["embarked"])

# View the encoded values
print(data[["embarked"]].head())

# Show the mapping
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("encoding mapping:", mapping)

   embarked
0         2
1         2
2         2
3         2
4         2
encoding mapping: {'C': 0, 'Q': 1, 'S': 2, nan: 3}


In [27]:
# Create a ticket_purchase_date column using today's date
data["ticket_purchase_date"] = pd.Timestamp.today()

# Ensure the column is in datetime format
data["ticket_purchase_date"] = pd.to_datetime(data["ticket_purchase_date"])

# Extract the year and month
data["purchase_year"] = data["ticket_purchase_date"].dt.year
data["purchase_month"] = data["ticket_purchase_date"].dt.month

# Display the results
print(data[["ticket_purchase_date", "purchase_year", "purchase_month"]].head())

        ticket_purchase_date  purchase_year  purchase_month
0 2026-07-23 13:48:11.567977           2026               7
1 2026-07-23 13:48:11.567977           2026               7
2 2026-07-23 13:48:11.567977           2026               7
3 2026-07-23 13:48:11.567977           2026               7
4 2026-07-23 13:48:11.567977           2026               7


In [28]:
  # Remove titles and convert names to lowercase
data["name"] = (
    data["name"]
    .str.replace(r",\s*(Mr|Mrs|Miss|Ms|Dr|Master|Rev|Col|Major|Capt|Lady|Sir|Don|Dona|Mme|Mlle|Jonkheer|Countess)\.?", "", regex=True)
    .str.strip()
    .str.lower()
)

# Display the cleaned names
print(data["name"].head())

0                         allen elisabeth walton
1                          allison hudson trevor
2                          allison helen loraine
3                allison hudson joshua creighton
4    allisons. hudson j c (bessie waldo daniels)
Name: name, dtype: object


In [29]:
# create age cats
data["age_category"] = pd.cut(
    data["age"],
    bins=[0, 12, 19, 64, float("inf")]
,
labels= ["child", "teen", "adult", "senior"]
)

print(data[["age", "age_category"]].head())
print(data["age_category"].value_counts())

       age age_category
0  29.0000        adult
1   0.9167        child
2   2.0000        child
3  30.0000        adult
4  25.0000        adult
age_category
adult     1071
teen       131
child       94
senior      13
Name: count, dtype: int64
